In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm

import concurrent.futures


import numpy as np
import pandas as pd
import ast
import glob
import pickle
import dask
import os
import itertools

import pickle

#from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from statsmodels.regression.rolling import RollingOLS

from tqdm.notebook import tqdm

from multiprocessing import Pool, cpu_count
from joblib import Parallel, delayed
#from tqdm import tqdm
from collections import Counter
from functools import reduce


import dask
import dask.dataframe as dd
from dask.distributed import Client
from dask.diagnostics import ProgressBar

#client = Client(n_workers=20, memory_limit="10GB", interface='lo')
from concurrent.futures import ThreadPoolExecutor

import dask_ml.cluster as dask_cluster

from pprint import pprint
import os

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

### Load TLGRF Benchmark Dataset

In [ ]:
benchmark_TLGRF_dataset = pd.read_parquet("../data/benchmark_tlgrf/")
benchmark_TLGRF_dataset["date"] = pd.to_datetime(benchmark_TLGRF_dataset["date"])
benchmark_TLGRF_dataset = benchmark_TLGRF_dataset[benchmark_TLGRF_dataset["log_rolled_cases"] >= np.log(21.1)]
benchmark_TLGRF_dataset = benchmark_TLGRF_dataset[benchmark_TLGRF_dataset["date"] <= "2022-12-31"]
benchmark_TLGRF_dataset = benchmark_TLGRF_dataset.sort_values(by=["fips","date"])
display(benchmark_TLGRF_dataset)

In [ ]:
TLGRF_MAE = benchmark_TLGRF_dataset.groupby("date").apply(lambda x: np.nanmean(abs(x["TLGRF_predicted_log_rolled_cases"]- x["shifted_log_rolled_cases"])))
TLGRF_RMSE = benchmark_TLGRF_dataset.groupby("date").apply(lambda x: np.sqrt(np.nanmean( (x["TLGRF_predicted_log_rolled_cases"]- x["shifted_log_rolled_cases"])**2 )))


In [ ]:
TLGRF_MAE

### Load Augmented DF Benchmark Dataset

In [ ]:
augmented_df = pd.read_parquet("../data/augmented_us_counties.parquet")
augmented_df["date"] = pd.to_datetime(augmented_df["date"])
augmented_df["fips"] = augmented_df["fips"].astype(int)
augmented_df["days_from_start"] = augmented_df["days_from_start"].astype(int)
augmented_df = augmented_df.sort_values(by=["fips","date"])

# Check for gaps
gt_columns = ["fips", "days_from_start", "date", "log_rolled_cases", "shifted_log_rolled_cases"]
augmented_df_gt = augmented_df[gt_columns]


### Load Benchmark Data of $wsize \in \{2,3,4,\dotsc,13,14\}$

In [ ]:
# Read per-window-size beta data from the consolidated pre-computed parquet
import re
bfw = pd.read_parquet('../data/benchmark_fixed_window/')
bfw['date'] = pd.to_datetime(bfw['date'])
beta_results = []
window_sizes = list(range(2, 15))
for fips in tqdm(bfw['fips'].unique()):
    for wsize in window_sizes:
        col = f'beta_wsize={wsize}'
        sub = bfw[bfw['fips']==fips][['fips','days_from_start',col]].dropna(subset=[col]).copy()
        if not sub.empty:
            beta_results.append(sub)


### Check for Duplicates

In [ ]:
pass  # fips/wsize mapping now handled by parquet loading


### Remove fips with too few entries

In [ ]:
sorted_beta_results = sorted(beta_results, key=lambda x: (int(x["fips"].unique()[0]), int(x.columns[2].split("=")[1])))


In [ ]:
pass

### Merge Results

In [ ]:
problematic_fips = {}
beta_result_dict = {window_size:[] for window_size in window_sizes}
for beta_df in tqdm(sorted_beta_results):
    fips = int(beta_df["fips"].unique()[0])
    window_size = int(beta_df.columns[2].split("=")[1])
    if fips not in problematic_fips.keys() and window_size in window_sizes:
        beta_result_dict[window_size].append(beta_df)

concatenated_beta_result_dict = {}
for window_size, beta_df_list in tqdm(beta_result_dict.items()):
    concatenated_beta_result_dict[window_size] = pd.concat(beta_df_list)
#updated_df.to_csv("TLGRF_w_Fixed_Windows.csv", index=False)

In [ ]:
concatenated_beta_result_dict

In [ ]:
beta_df_big = pd.DataFrame()
#for window_size, fips_beta_df in tqdm(concatenated_beta_result_dict.items()):
#    if not beta_df_big.shape[0]:
#        beta_df_big = fips_beta_df.copy()
#        continue
#    beta_df_big = pd.merge(beta_df_big, fips_beta_df, on="fips", how="outer")

beta_df_big = reduce(lambda left, right: pd.merge(left, right, on=['fips','days_from_start'], how='outer'), concatenated_beta_result_dict.values())


updated_df = pd.merge(augmented_df_gt, beta_df_big,on=['fips','days_from_start'], how="outer").sort_values(by=["fips", 'days_from_start'])
filtered_updated_df = updated_df[updated_df["date"] <= "2022-12-31"]
#filtered_updated_df = pd.merge(filtered_updated_df, augmented_df_gt, on=["fips","days_from_start"], how="left")

display(filtered_updated_df)

In [ ]:
predictor_columns = ["beta_wsize={}".format(window_size) for window_size in window_sizes]

In [ ]:
MAE_df = pd.DataFrame()
RMSE_df = pd.DataFrame()
MAE_df["r_TLGRF"] = TLGRF_MAE
RMSE_df["r_TLGRF"] = TLGRF_RMSE
for predictor_column in tqdm(predictor_columns):
    predictor_MAE = filtered_updated_df.groupby("date").apply(lambda x: np.nanmean(abs(x[predictor_column]*7+x["log_rolled_cases"]- x["shifted_log_rolled_cases"])))
    predictor_RMSE = filtered_updated_df.groupby("date").apply(lambda x: np.sqrt(np.nanmean( (x[predictor_column]*7+x["log_rolled_cases"] - x["shifted_log_rolled_cases"])**2 )))
    MAE_df[predictor_column] = predictor_MAE
    RMSE_df[predictor_column] = predictor_RMSE

In [ ]:
MAE_df

In [ ]:
RMSE_df

In [ ]:
metrics_comparison_df = pd.DataFrame()

row_names = ["TLGRF"] + ["Fixed Window {}".format(window_size) for window_size in window_sizes]

metrics_comparison_df["MAE"] = MAE_df.median()
metrics_comparison_df["RMSE"] = RMSE_df.median()
metrics_comparison_df.index = row_names
metrics_comparison_df = metrics_comparison_df[::-1]
metrics_comparison_df

In [ ]:
latex_table = metrics_comparison_df.to_latex(column_format='c'*len(metrics_comparison_df.columns), float_format='%.3f', escape=False)
print(latex_table)

In [ ]:
fig = plt.figure(figsize=(18,10))

ax1 = fig.add_subplot(2, 1, 1)
# Create the bottom left subplot
ax2 = fig.add_subplot(2, 2, 3)
# Create the bottom right subplot
ax3 = fig.add_subplot(2, 2, 4)

# Adjust the spacing between subplots
#fig.subplots_adjust(hspace=0.3)

wsizes = [2,7,14]
colors_lm = ["red", "blue", "xkcd:dark turquoise", "magenta"]
colors_lm = list(reversed(colors_lm))

linestyles_lm = ["-","dotted","dashed","dotted"]
linestyles_lm = list(reversed(linestyles_lm))


plot_columns =  ["r_TLGRF"] + ["beta_wsize={}".format(window_size) for window_size in [2,7,14]]
plot_columns = reversed(plot_columns)

plot_row_names = ["TLGRF"] + ["Fixed Window {}".format(window_size) for window_size in [2,7,14]]
plot_row_names = list(reversed(plot_row_names))

#plot_columns =  ["r_TLGRF"] + ["beta_wsize={}".format(window_size) for window_size in [2]]
ax1_line_handles = []
for i,plot_column in tqdm(enumerate((plot_columns))):
    ax1_line_handles.append(ax1.plot(MAE_df[plot_column], label=plot_row_names[i], color=colors_lm[i], linestyle=linestyles_lm[i]))
    ax2.plot(MAE_df[plot_column], label=plot_row_names[i], color=colors_lm[i], linestyle=linestyles_lm[i])
    ax3.plot(MAE_df[plot_column], label=plot_row_names[i], color=colors_lm[i], linestyle=linestyles_lm[i])
    
handles = [handle[0] for handle in ax1_line_handles]
labels = [handle[0].get_label() for handle in ax1_line_handles]
locator = mdates.DayLocator(interval=15)
formatter = mdates.DateFormatter('%Y-%m-%d')



ax1.set_xlabel("Date")
ax1.set_ylabel("MAE")
ax1.set_title("(A): MAE of Fixed Window Sizes vs TLGRF")
ax1.set_xlim(pd.to_datetime("2020-03-15"), pd.to_datetime("2023-01-01"))
ax1.set_ylim(0,0.6)
ax1.legend(handles, labels, loc='upper right')

ax2.set_xlabel("Date")
ax2.set_ylabel("MAE")
ax2.set_title("(B): Initial Lockdown Period")
ax2.set_xlim(pd.to_datetime("2020-04-30"), pd.to_datetime("2020-08-01"))
ax2.set_ylim(0,0.6)
#ax2.xaxis.set_major_locator(locator)
#ax2.xaxis.set_major_formatter(formatter)


ax3.set_xlabel("Date")
ax3.set_ylabel("MAE")
ax3.set_title("(C): Delta Variant")
ax3.set_xlim(pd.to_datetime("2021-07-01"), pd.to_datetime("2021-08-31"))
ax3.set_ylim(0,0.6)
ax3.xaxis.set_major_locator(locator)
ax3.xaxis.set_major_formatter(formatter)


#plt.legend()
#fig.legend(handles, labels, loc='upper ')

# Adjust the layout
plt.tight_layout()
plt.savefig("new_lm_grf_mae_together.png")

plt.show()

In [ ]:
[handle[0].get_label() for handle in ax1_line_handles]

In [ ]:
fig = plt.figure(figsize=(18,10))

ax1 = fig.add_subplot(2, 1, 1)
# Create the bottom left subplot
ax2 = fig.add_subplot(2, 2, 3)
# Create the bottom right subplot
ax3 = fig.add_subplot(2, 2, 4)

# Adjust the spacing between subplots
#fig.subplots_adjust(hspace=0.3)

plot_columns =  ["r_TLGRF"] + ["beta_wsize={}".format(window_size) for window_size in [2,7,14]]
plot_columns = reversed(plot_columns)

plot_row_names = ["TLGRF"] + ["Fixed Window {}".format(window_size) for window_size in [2,7,14]]
plot_row_names = list(reversed(plot_row_names))

colors_lm = ["red", "blue", "xkcd:dark turquoise", "magenta"]
colors_lm = list(reversed(colors_lm))

linestyles_lm = ["-","dotted","dashed","dotted"]
linestyles_lm = list(reversed(linestyles_lm))


plot_columns =  ["r_TLGRF"] + ["beta_wsize={}".format(window_size) for window_size in [2,7,14]]
plot_columns = reversed(plot_columns)

plot_row_names = ["TLGRF"] + ["Fixed Window {}".format(window_size) for window_size in [2,7,14]]
plot_row_names = list(reversed(plot_row_names))


#plot_columns =  ["r_TLGRF"] + ["beta_wsize={}".format(window_size) for window_size in [2]]
ax1_line_handles = []
for i,plot_column in tqdm(enumerate((plot_columns))):
    ax1_line_handles.append(ax1.plot(RMSE_df[plot_column], label=plot_row_names[i], color=colors_lm[i], linestyle=linestyles_lm[i]))
    ax2.plot(RMSE_df[plot_column], label=plot_row_names[i], color=colors_lm[i], linestyle=linestyles_lm[i])
    ax3.plot(RMSE_df[plot_column], label=plot_row_names[i], color=colors_lm[i], linestyle=linestyles_lm[i])
    
handles = [handle[0] for handle in ax1_line_handles]
labels = [handle[0].get_label() for handle in ax1_line_handles]
locator = mdates.DayLocator(interval=15)
formatter = mdates.DateFormatter('%Y-%m-%d')



ax1.set_xlabel("Date")
ax1.set_ylabel("RMSE")
ax1.set_title("(A): RMSE of Fixed Window Sizes vs TLGRF")
ax1.set_xlim(pd.to_datetime("2020-03-15"), pd.to_datetime("2023-01-01"))
ax1.set_ylim(0,1.0)
ax1.legend(handles, labels, loc='upper right')

ax2.set_xlabel("Date")
ax2.set_ylabel("RMSE")
ax2.set_title("(B): Initial Lockdown Period")
ax2.set_xlim(pd.to_datetime("2020-04-30"), pd.to_datetime("2020-08-01"))
ax2.set_ylim(0,1.0)
#ax2.xaxis.set_major_locator(locator)
#ax2.xaxis.set_major_formatter(formatter)


ax3.set_xlabel("Date")
ax3.set_ylabel("RMSE")
ax3.set_title("(C): Delta Variant")
ax3.set_xlim(pd.to_datetime("2021-07-01"), pd.to_datetime("2021-08-31"))
ax3.set_ylim(0,1.0)
ax3.xaxis.set_major_locator(locator)
ax3.xaxis.set_major_formatter(formatter)


#plt.legend()
#fig.legend(handles, labels, loc='upper ')

# Adjust the layout
plt.tight_layout()
plt.savefig("new_lm_grf_rmse_together.png")

plt.show()

### Save the MAE and RMSE Dataframes

In [ ]:
RMSE_df.to_csv("Fixed_windows_RMSE_df.csv")
MAE_df.to_csv("Fixed_windows_MAE_df.csv")


In [ ]:
MAE_df[MAE_df.index <= "2021-09-12"].median()

In [ ]:
RMSE_df[RMSE_df.index <= "2021-09-12"].median()

In [ ]:
pass

In [ ]:
filtered_updated_df.to_csv("Fixed_windows_all_beta.csv", index=False)